In [2]:
train_dir = "/content/drive/MyDrive/train"

In [3]:
test_dir = "/content/drive/MyDrive/test"

In [7]:
import numpy as np
from transformers import ViTFeatureExtractor
from PIL import Image,ImageOps
model = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')




/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [9]:
img = Image.open(r"/content/drive/MyDrive/train/NORMAL/IM-0115-0001.jpeg")
img = img.resize((100,100))
img = ImageOps.colorize(img, black="black", white="white") #only Pneumonia data
img =model(images=img)
img = img["pixel_values"]
img = np.array(img)
img = img.flatten()
img.shape


(150528,)

In [10]:
X_train = []
Y_train = []
counter=1

import os
import numpy as np
for i in os.listdir(train_dir):
    print(i)
    for j in os.listdir(train_dir + "/" +  i ):
        try:
            print(train_dir + "/" + i + "/" + j)
            img = Image.open(train_dir + "/" + i + "/" + j)
            img = img.resize((50,50))
            img = ImageOps.colorize(img, black="black", white="white") #only for Pneumonia data
            img =model(images=img)
            img = img["pixel_values"]
            img = np.array(img)
            X_train.append(img.flatten())
            Y_train.append(i)
        except:
             print(counter)
             counter = counter + 1
print(counter)


流式输出内容被截断，只能显示最后 5000 行内容。
/content/drive/MyDrive/train/PNEUMONIA/person707_virus_1305.jpeg
/content/drive/MyDrive/train/PNEUMONIA/person705_virus_1302.jpeg
/content/drive/MyDrive/train/PNEUMONIA/person704_bacteria_2603.jpeg
... (4993 lines of per-image progress output removed) ...
/content/drive/MyDrive/train/NORMAL/IM-0523-0001-0001.jpeg
/content/drive/MyDrive/train/NORMAL/IM-0525-0001-0001.jpeg
/content/drive/MyDrive/train/NORMAL/IM-0526-0001.jpeg
284


In [11]:
X_test = []
Y_test = []
counter=1

import os
import numpy as np
for i in os.listdir(test_dir):
    print(i)
    for j in os.listdir(test_dir + "/" +  i ):
        try:
            print(train_dir + "/" + i + "/" + j)
            img = Image.open(test_dir + "/" + i + "/" + j)
            img = img.resize((100,100))
            img = ImageOps.colorize(img, black="black", white="white")
            img =model(images=img)
            img = img["pixel_values"]
            img = np.array(img)
            X_test.append(img.flatten())
            Y_test.append(i)
        except Exception as e:
             print("error is ", e, "counter is ", counter)
             counter = counter + 1


PNEUMONIA
/content/drive/MyDrive/train/PNEUMONIA/person100_bacteria_477.jpeg
/content/drive/MyDrive/train/PNEUMONIA/person100_bacteria_482.jpeg
/content/drive/MyDrive/train/PNEUMONIA/person101_bacteria_486.jpeg
... (618 lines of per-image progress output removed) ...
/content/drive/MyDrive/train/NORMAL/NORMAL2-IM-0374-0001-0001.jpeg
/content/drive/MyDrive/train/NORMAL/NORMAL2-IM-0374-0001.jpeg
/content/drive/MyDrive/train/NORMAL/NORMAL2-IM-0372-0001.jpeg
/content/drive/MyDrive/train/NORMAL/NORMAL2-IM-0370-0001.jpeg


In [12]:
from sklearn.metrics import confusion_matrix
from sklearn import linear_model
model = linear_model.LogisticRegression().fit(X_train,Y_train)
pred = model.predict(X_test)
cm = confusion_matrix(Y_test,pred)
print("accuracy is ", (cm[0,0]+cm[1,1])/(sum(sum(cm))))


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


accuracy is  0.7532051282051282


In [13]:
from sklearn import ensemble
model = ensemble.RandomForestClassifier().fit(X_train,Y_train)
pred = model.predict(X_test)
cm = confusion_matrix(Y_test,pred)
print("accuracy is ", (cm[0,0]+cm[1,1])/(sum(sum(cm))))


accuracy is  0.7596153846153846
